Question 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [4]:
TARGET_COL = "price"
DROP_COLS = ["id", "date", "zipcode", "Unnamed: 0"]

def load_and_prepare(path: str):
    df = pd.read_csv(path)

    #divide price by 1000
    y = (df[TARGET_COL].astype(float) / 1000.0).values

    #drop target + excluded columns
    X = df.drop(columns=[TARGET_COL] +[c for c in DROP_COLS if c in df.columns],
                errors="ignore")

    #keep numeric features only
    X = X.select_dtypes(include=[np.number])
    return X, y

In [9]:
train_path = "train.csv"
test_path  = "test.csv"

X_train_df, y_train = load_and_prepare(train_path)
X_test_df,  y_test  = load_and_prepare(test_path)

X_train_df, X_test_df = X_train_df.align(X_test_df,join="inner",axis = 1)
feature_names = X_train_df.columns.tolist()

imputer = SimpleImputer(strategy = "median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train_df))
X_test  = scaler.transform(imputer.transform(X_test_df))

model = LinearRegression()
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred  = model.predict(X_test)

train_mse = mean_squared_error(y_train,train_pred)
test_mse  = mean_squared_error(y_test, test_pred)
train_r2  = r2_score(y_train, train_pred)
test_r2   = r2_score(y_test, test_pred)

print("Train MSE:", train_mse, "R2:", train_r2)
print("Test MSE:", test_mse,  "R2:", test_r2)

#coefficients
coef_table = pd.DataFrame({"feature": feature_names, "coef": model.coef_})
coef_table["abs_coef"] = coef_table["coef"].abs()
coef_table = coef_table.sort_values("abs_coef",
                                    ascending=False).drop(columns=["abs_coef"])
print(coef_table.to_string(index=False))
print("Intercept:", model.intercept_)


Train MSE: 31486.167775794882 R2: 0.7265334318706018
Test MSE: 57628.154705670415 R2: 0.6543560876120953
      feature       coef
        grade  92.231475
          lat  78.375737
     yr_built -67.643117
   waterfront  63.742900
  sqft_living  56.748837
   sqft_above  48.290089
         view  48.200109
sqft_living15  45.577658
sqft_basement  27.137032
    bathrooms  18.527633
 yr_renovated  17.271380
    condition  12.964269
   sqft_lot15 -12.930091
     bedrooms -12.521962
     sqft_lot  10.881868
       floors   8.043721
         long  -1.035203
Intercept: 520.414834000001


**Question 2.2: Interpret the results in your own words. Which features contribute mostly to the linear regression model? Is the model fitting the data
well? How large is the model error? How do the training and testing MSE relate?**             
By the coefficients, the biggest contributors are grade, lat, yr_built, waterfront, sqft_living, sqft_above, and view. So, construction/quality, location, and house size matter the most. The model fits the data moderately well. The R squared value is ~0.654 which means the linear model explains about 65% of the variation in house prices. That is an okay score for a simple linear model but it can definitely be improved. The test RMSE is the square root of the MSE which is around 240.059 which means the average error is around $240,059. The test MSE (57628) is higher than the train MSE (31486). That gap indicates that the model generalizes reasonably but there is some overfitting or the test split is a bit harder.